In [21]:
import os
import sys

import joblib
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

sys.path.insert(0, '../../..')
from utils.plotting import plot_movement_data
from utils.constants import Configs
from utils.loso_pipeline import run_loso_evaluation
from utils.loso_models.iforest_detector import IForestDetector
from utils.metrics import show_feature_extraction_metrics
from utils.bootstrap_utils import compute_bootstrap_ci_all_levels
from utils.helper_functions import save_loso_results
from utils.helper_functions import append_experiment_record
from utils.loso_pipeline import aggregate_to_trial_level, derive_trial_level_gt, summarize_level
from utils.feature_selection import make_corr_filter

TARGET_MOVEMENT = 'Move'
ALGORITHM_NAME = 'IForest'
MODEL_REPRESENTATION = 'FeatureBased'

N_BAYES_TRIALS = 40 
EXTRACT_SCORES = True

In [22]:
# Load the processed data from the csv
cache_path = Configs.ICATER_WRIST_ONLY_FILTERED_DATA_DIR
if not os.path.exists(cache_path):
    raise FileNotFoundError(f"Processed data not found at {cache_path}. Please run the data processing notebook first.")

df = pd.read_csv(cache_path)
df = df.sort_values(by=['sub', 'repID', 'movID', 'step']).reset_index(drop=True)

In [23]:
# Load the known outliers from the csv
outliers_csv = pd.read_csv(Configs.ICATER_OUTLIER_FILES).drop_duplicates(subset=['sub', 'repID', 'movID'])
total_known_outliers = len(outliers_csv)

# Filter the outliers for the target movement
outliers_csv = outliers_csv[outliers_csv['movLabel'] == TARGET_MOVEMENT].reset_index(drop=True)
total_target_outliers = len(outliers_csv)

print(f"Total known outliers: {total_known_outliers}")
print(f"Total target outliers: {total_target_outliers}")
outliers_csv.head()

Total known outliers: 18
Total target outliers: 5


,sub,repID,movID,movLabel
0,2,2,9,Move
1,4,1,9,Move
2,7,1,9,Move
3,11,1,9,Move
4,12,1,9,Move


### Load Features from csv

In [24]:
cache_path = Configs.wrist_only_feature_extraction_features(TARGET_MOVEMENT)
if not os.path.exists(cache_path):
    raise FileNotFoundError(f"Features for {TARGET_MOVEMENT} not found at {cache_path}. Please run the feature extraction notebook first.")

features_df = joblib.load(cache_path)
features_df = features_df.sort_values(by=['sub', 'repID', 'movID']).reset_index(drop=True)
X_features = features_df[Configs.FEATURE_COLS].to_numpy()

assert not np.isnan(X_features).any(), "NaN in features"

In [25]:
if total_target_outliers == 0:
    raise RuntimeError(
        f"'{TARGET_MOVEMENT}' Does not have any known positive cases at the movement level. "
    )

## Isolation Forest

In [26]:
total_trials_unique = features_df.drop_duplicates(subset=['sub', 'repID'])
total_trials = len(total_trials_unique)
total_samples = len(features_df)
known_outlier_rate = (total_target_outliers / total_samples) if total_samples > 0 else 0.0
percentagem_conhecida = known_outlier_rate * 100

print(f"--- Preparation (Feature-Based) ---")
print(f"Total samples analyzed ({TARGET_MOVEMENT}): {total_samples}")
print(f"Total unique repetitions analyzed ({TARGET_MOVEMENT}): {total_trials}")
print(f"Minimum confirmed outlier rate in CSV (sub + repID): {percentagem_conhecida:.2f}%")

--- Preparation (Feature-Based) ---
Total samples analyzed (Move): 95
Total unique repetitions analyzed (Move): 95
Minimum confirmed outlier rate in CSV (sub + repID): 5.26%


### Grid Search

In [ ]:
FIXED_SEED = 42

# Cannot use itertools when I'm using Bayesian optimization
param_space = {
    'n_estimators': ('int', 50, 500),
    'max_samples': ('categorical', ['auto', 0.3, 0.5, 0.7, 0.9]),
    'contamination': ('categorical', Configs.CONTAMINATION_GRID),
    'random_state': FIXED_SEED
}

loso_results = run_loso_evaluation(
    detector_class=IForestDetector, 
    param_grid=param_space,
    data_matrix=X_features,                     # all features
    metadata_df=features_df[Configs.META_COLS], # same rows as X_features
    known_outliers_df=outliers_csv,
    search_type='bayesian',
    n_bayes_trials=N_BAYES_TRIALS,
    metric_type='euclidean',
    scale_features=True,
    compute_trial_level=False,
    extract_scores=EXTRACT_SCORES,
    print_results=False,
    feature_filter=make_corr_filter(Configs.FEATURE_COLS, Configs.FEATURES_CORRELATION_THRESHOLD, Configs.PREFER_DROP),
)

LOSO Cross-Validation with 19 subjects (5 known positive movements)...
Evaluating Subject: 1 | Evaluating Subject: 2 | Evaluating Subject: 3 | Evaluating Subject: 4 | Evaluating Subject: 5 | Evaluating Subject: 6 | Evaluating Subject: 7 | Evaluating Subject: 8 | Evaluating Subject: 9 | Evaluating Subject: 10 | Evaluating Subject: 11 | Evaluating Subject: 12 | Evaluating Subject: 13 | Evaluating Subject: 14 | Evaluating Subject: 15 | Evaluating Subject: 16 | Evaluating Subject: 17 | Evaluating Subject: 18 | Evaluating Subject: 19 | 

In [28]:
# n_estimators: Número de árvores na floresta. Mais árvores podem melhorar a detecção, mas aumentam o tempo de treino.
# contamination: Auto é uma boa opção para começar, pois estima a proporção de outliers com base nos dados.
# random_state was set with the answer to the life, universe and everything (for reproducibility)
# max_samples: 'auto' $$max\_samples = \min(256, n\_samples)$$

best_params = loso_results['Best_Params']
n_estimators = best_params['n_estimators']
contamination = best_params['contamination']
max_samples = best_params['max_samples']

final_scaler = StandardScaler()
X_features_scaled_full = final_scaler.fit_transform(X_features)

##  Report Seed Variance

In [29]:
# Extract the exact hyperparameter dictionary used for each specific subject in the original run
fold_params = {
    fold['subject_out']: fold['params'].copy()
    for fold in loso_results['Fold_History']
}

In [ ]:
# --- Variance Evaluation for Stochastic Model ---
test_seeds = [10, 42, 123, 999, 2026]
f2_scores_seeds = []
precision_scores = []
recall_scores = []
auprc_scores = []
mcc_scores = []
all_seed_results = []

for seed in test_seeds:
    # Single configuration grid with changing seed
    param_space_seed = {
        'n_estimators': ('int', 50, 500),
        'max_samples': ('categorical', ['auto', 0.3, 0.5, 0.7, 0.9]),
        'contamination': ('categorical', Configs.CONTAMINATION_GRID),
        'random_state': seed
    }

    seed_results = run_loso_evaluation(
        detector_class=IForestDetector, 
        param_grid=param_space_seed,
        data_matrix=X_features,                     # all features
        metadata_df=features_df[Configs.META_COLS], # same rows as X_features
        known_outliers_df=outliers_csv,
        search_type='bayesian',
        n_bayes_trials=N_BAYES_TRIALS,
        metric_type='euclidean',
        fixed_fold_params=fold_params,
        override_random_state=seed, 
        scale_features=True,
        compute_trial_level=False,
        extract_scores=EXTRACT_SCORES,
        print_results=False,
        feature_filter=make_corr_filter(Configs.FEATURE_COLS, Configs.FEATURES_CORRELATION_THRESHOLD, Configs.PREFER_DROP),
    )

    all_seed_results.append(seed_results) 
    f2_scores_seeds.append(seed_results['Movement']['F2-Score'])
    precision_scores.append(seed_results['Movement']['Precision'])
    recall_scores.append(seed_results['Movement']['Recall'])
    mcc_scores.append(seed_results['Movement']['MCC'])
    
    if EXTRACT_SCORES:
        auprc_scores.append(seed_results['Movement']['AUPRC'])
    
print("\n\n-- IForest Stability Results (5 seeds) ---")
print(f"F2-Score:  {np.mean(f2_scores_seeds):.2f}% ± {np.std(f2_scores_seeds):.2f}%")
print(f"Precision: {np.mean(precision_scores):.2f}% ± {np.std(precision_scores):.2f}%")
print(f"Recall:    {np.mean(recall_scores):.2f}% ± {np.std(recall_scores):.2f}%")
print(f"MCC:       {np.mean(mcc_scores):.2f}% ± {np.std(mcc_scores):.2f}%")
print(f"AUPRC:     {np.mean(auprc_scores):.2f}% ± {np.std(auprc_scores):.2f}%")
print("-------------------------------------------")

LOSO Cross-Validation with 19 subjects (5 known positive movements)...
Evaluating Subject: 1 | Evaluating Subject: 2 | Evaluating Subject: 3 | Evaluating Subject: 4 | Evaluating Subject: 5 | Evaluating Subject: 6 | Evaluating Subject: 7 | Evaluating Subject: 8 | Evaluating Subject: 9 | Evaluating Subject: 10 | Evaluating Subject: 11 | Evaluating Subject: 12 | Evaluating Subject: 13 | Evaluating Subject: 14 | Evaluating Subject: 15 | Evaluating Subject: 16 | Evaluating Subject: 17 | Evaluating Subject: 18 | Evaluating Subject: 19 | LOSO Cross-Validation with 19 subjects (5 known positive movements)...
Evaluating Subject: 1 | Evaluating Subject: 2 | Evaluating Subject: 3 | Evaluating Subject: 4 | Evaluating Subject: 5 | Evaluating Subject: 6 | Evaluating Subject: 7 | Evaluating Subject: 8 | Evaluating Subject: 9 | Evaluating Subject: 10 | Evaluating Subject: 11 | Evaluating Subject: 12 | Evaluating Subject: 13 | Evaluating Subject: 14 | Evaluating Subject: 15 | Evaluating Subject: 16 | E

In [31]:
oof_movement_per_seed = [sr['OOF_Movement'] for sr in all_seed_results]

reference_keys = oof_movement_per_seed[0][['sub', 'repID', 'movID', 'true_is_outlier']].reset_index(drop=True)
for oof in oof_movement_per_seed[1:]:
    keys = oof[['sub', 'repID', 'movID', 'true_is_outlier']].reset_index(drop=True)
    assert reference_keys.equals(keys), "As linhas de OOF_Movement divergem entre seeds."

stacked_preds = np.stack([oof['pred_is_outlier'].values for oof in oof_movement_per_seed], axis=0)
consensus_pred = (stacked_preds.mean(axis=0) >= 0.5).astype(int)

if all(oof['anomaly_score'].notna().all() for oof in oof_movement_per_seed):
    stacked_scores = np.stack([oof['anomaly_score'].values for oof in oof_movement_per_seed], axis=0)
    consensus_score = stacked_scores.mean(axis=0)
else:
    consensus_score = np.full(len(reference_keys), np.nan)

oof_movement_consensus = reference_keys.copy()
oof_movement_consensus['pred_is_outlier'] = consensus_pred
oof_movement_consensus['anomaly_score'] = consensus_score

trial_outliers_gt = derive_trial_level_gt(outliers_csv)
oof_trial_consensus = aggregate_to_trial_level(oof_movement_consensus, trial_outliers_gt, score_col='anomaly_score')

movement_summary_consensus = summarize_level(oof_movement_consensus, 'movement', len(outliers_csv))
trial_summary_consensus = summarize_level(oof_trial_consensus, 'trial', len(trial_outliers_gt))

In [32]:
loso_results_consensus = dict(loso_results)
loso_results_consensus['Movement'] = movement_summary_consensus
# loso_results_consensus['Trial'] = trial_summary_consensus
loso_results_consensus['OOF_Movement'] = oof_movement_consensus
# loso_results_consensus['OOF_Trial'] = oof_trial_consensus
loso_results_consensus['Blind_Y_True'] = oof_movement_consensus['true_is_outlier'].values
loso_results_consensus['Blind_Y_Pred'] = oof_movement_consensus['pred_is_outlier'].values

print("--- IForest (Feature-Based, Seed-Robust Consensus) -- nível de MOVIMENTO ---")
for k in ['Precision', 'Recall', 'F1-Score', 'F2-Score', 'MCC', 'AUPRC', 'AUROC', 'Rejection_Rate']:
    v = movement_summary_consensus.get(k, np.nan)
    print(f"  {k}: {v:.4f}" if isinstance(v, float) and not np.isnan(v) else f"  {k}: {v}")

--- IForest (Feature-Based, Seed-Robust Consensus) -- nível de MOVIMENTO ---
  Precision: 1.0000
  Recall: 1.0000
  F1-Score: 1.0000
  F2-Score: 1.0000
  MCC: 1.0000
  AUPRC: 1.0000
  AUROC: 1.0000
  Rejection_Rate: 5.2632


## Bootstrap CI interval 95%

In [33]:
ci_results_consensus = compute_bootstrap_ci_all_levels(loso_results_consensus, compute_scores=EXTRACT_SCORES, seed=42)

In [34]:
save_loso_results(
    model_name=ALGORITHM_NAME,
    representation=MODEL_REPRESENTATION,
    target_movement=TARGET_MOVEMENT,
    loso_results=loso_results_consensus,
)

append_experiment_record(
    summary_path=Configs.WRIST_ONLY_FEATURE_EXTRACTION_SUMMARY_FILE,
    phase=TARGET_MOVEMENT,
    representation=MODEL_REPRESENTATION,
    algorithm=ALGORITHM_NAME,
    total_samples=total_samples,
    loso_results=loso_results_consensus,
    bootstrap_results=ci_results_consensus,
)


✅ Experiment Summary for IForest - FeatureBased - Move - 95 Movements (MOVEMENT-level):
Unique Outliers Detected: 5
Rejection Rate: 5.26%
Known Positives in Ground Truth: 5
------------------------------
Total True Positives (Global):  5
Total False Positives (Global): 0
Total False Negatives (Global): 0
Total True Negatives (Global):  90
------------------------------
Blind Precision: 100.00 [100.00–100.00]
Blind Recall:    100.00 [100.00–100.00]
Blind F1-Score:  100.00
Blind F2-Score:  100.00 [100.00–100.00]
Blind MCC:       1.00 [1.00–1.00]
Blind AUPRC:  100.00 [100.00–100.00]
Blind AUROC:   100.00

Total Tuning Time (19 Subjects): 494.5908s
Total Inference Time (19 Subjects): 2.7659s

Overall Best Parameters (Most frequent across folds):
   {'contamination': 0.05, 'max_samples': 0.3, 'n_estimators': 68, 'random_state': 42}
Occurrences: 14


### Train Model
Aqui não existe um treino de um modelo final, mas sim a agregação do voto maioritário do LOSO + teste de Seeds.

In [35]:
consensus_outlier_keys = set(
    zip(oof_trial_consensus[oof_trial_consensus['pred_is_outlier'] == 1]['sub'],
        oof_trial_consensus[oof_trial_consensus['pred_is_outlier'] == 1]['repID'])
)
print(f"✅ {len(consensus_outlier_keys)} trials marcados como outlier pelo consenso.")

✅ 5 trials marcados como outlier pelo consenso.


In [36]:
features_df['anomaly'] = features_df.apply(
    lambda row: -1 if (row['sub'], row['repID']) in consensus_outlier_keys else 1,
    axis=1
)

n_outliers =  len(consensus_outlier_keys)
total_trials_consensus = len(oof_trial_consensus)
correct_trash_pct = (n_outliers / total_trials_consensus) * 100 if total_trials_consensus > 0 else 0.0
n_unique_outliers = n_outliers

print(f"Unique Outlier Trials Detected (Consensus): {n_unique_outliers}")
print(f"Rejection Rate (Consensus): {correct_trash_pct:.2f}%")

outliers_details = features_df[features_df['anomaly'] == -1][['sub', 'repID', 'movID']]

Unique Outlier Trials Detected (Consensus): 5
Rejection Rate (Consensus): 5.26%


In [37]:
features_df['pred_is_outlier'] = (features_df['anomaly'] == -1).astype(int)
trial_level_df = features_df.groupby(['sub', 'repID'])['pred_is_outlier'].max().reset_index()

gt_keys = set(zip(outliers_csv['sub'], outliers_csv['repID']))

trial_level_df['true_is_outlier'] = trial_level_df.apply(
    lambda row: 1 if (row['sub'], row['repID']) in gt_keys else 0,
    axis=1
)

y_true_final = trial_level_df['true_is_outlier'].values
y_pred_final = trial_level_df['pred_is_outlier'].values

In [38]:
cols_to_extract = ['sub', 'repID', 'movID']
outliers_metadata_df = features_df.loc[outliers_details.index, cols_to_extract].copy()
print("Quem são os outliers reais?")
print(outliers_metadata_df)

Quem são os outliers reais?
    sub  repID  movID
6     2      2      9
15    4      1      9
30    7      1      9
50   11      1      9
55   12      1      9


In [39]:
identified_unique = outliers_metadata_df.drop_duplicates(subset=['sub', 'repID'])

matched_outliers = pd.merge(identified_unique, outliers_csv, on=['sub', 'repID'], how='inner')

missing_outliers = pd.merge(outliers_csv, identified_unique, on=['sub', 'repID'], how='left', indicator=True)
missing_outliers = missing_outliers[missing_outliers['_merge'] == 'left_only']

metrics = show_feature_extraction_metrics(
    features_df,
    outliers_csv,
    identified_unique,
    matched_outliers,
    missing_outliers
)

--- Confusion Matrix (Based on sub+repID) ---
True Positives (Identified and in CSV) : 5
False Positives (Identified, not in CSV): 0
False Negatives (In CSV, not identified): 0
True Negatives (Normal, stayed normal): 90
--------------------------------------------------
Missing outliers (In CSV, not identified):
Empty DataFrame
Columns: [sub, repID]
Index: []

Precision (False Positives): 5/5 (100.00%)
Recall (True Positive Rate): 5/5 (100.00%)
F1-Score: 100.00%
F2-Score: 100.00%


In [ ]:
outlier_ids = features_df[features_df['anomaly'] == -1][['sub', 'repID', 'movID']]

# Create a "unique key" to simplify filtering
features_df['key'] = features_df['sub'].astype(str) + '_' + features_df['repID'].astype(str) + '_' + features_df['movID'].astype(str)
outlier_keys = features_df[features_df['anomaly'] == -1]['key'].tolist()

# Return to the original DataFrame for plotting
# Create a temporary key in the original df as well
df['temp_key'] = df['sub'].astype(str) + '_' + df['repID'].astype(str) + '_' + df['movID'].astype(str)

# Separate visually
# Note: This is only for plotting; it does not permanently alter the original df if you do not want it to
df_outliers = df[df['temp_key'].isin(outlier_keys) & (df['movLabel'] == TARGET_MOVEMENT)]
df_clean = df[~df['temp_key'].isin(outlier_keys) & (df['movLabel'] == TARGET_MOVEMENT)]

plot_movement_data(
    clean_data=df_clean,
    outlier_data=df_outliers,
    target_movement=TARGET_MOVEMENT,
    algorithm_name=ALGORITHM_NAME,
    save_fig=Configs.SAVE_PLOTS,
    model_representation=MODEL_REPRESENTATION,
    save_path=Configs.DESKTOP_PATH
)

SyntaxError: keyword argument repeated: model_representation (1236292763.py, line 24)

In [ ]:
df.drop(columns=['temp_key'], inplace=True)